In [1]:
import numpy as np
from collections import Counter
import torch.nn
import pandas as pd
import matplotlib.pyplot as plt
import re

In [7]:
# re run with cleaned data
df = pd.read_csv('../resume_data_cleaned.csv')

In [8]:
df.head()

,matched_score,resume_text,job_text,combined_text,num_skills,num_degrees,num_positions,experience_years,age_min,age_max,skill_jaccard,job_position_name,educational_requirements,skills_required,skills,degree_names,positions
0,0.850000,big data analytics working and database wareho...,senior software engineer b.sc in computer scie...,big data analytics working and database wareho...,21,1,1,1.0,NaN,NaN,0.0,senior software engineer,b.sc in computer science & engineering from a ...,NaN,"[""big data"", ""hadoop"", ""hive"", ""python"", ""mapr...","[""b.tech""]","[""big data analyst""]"
1,0.750000,fresher looking to join as a data analyst and ...,machine learning (ml) engineer m.sc in compute...,fresher looking to join as a data analyst and ...,10,2,1,5.0,NaN,NaN,0.0,machine learning (ml) engineer,m.sc in computer science & engineering or in a...,NaN,"[""data analysis"", ""data analytics"", ""business ...","[""b.sc (maths)"", ""m.sc (science) (statistics)""]","[""business analyst""]"
2,0.416667,software development machine learning deep lea...,"executive/ senior executive- trade marketing, ...",software development machine learning deep lea...,14,1,1,3.0,NaN,NaN,0.0,"executive/ senior executive- trade marketing, ...",master of business administration (mba),brand promotion\r\ncampaign management\r\nfiel...,"[""software development"", ""machine learning"", ""...","[""b.tech""]","[""software developer (machine learning enginee..."
3,0.760000,to obtain a position in a fast-paced business ...,business development executive bachelor/honors...,to obtain a position in a fast-paced business ...,36,1,6,3.0,22.0,30.0,0.0,business development executive,bachelor/honors,fast typing skill\r\nieltsinternet browsing & ...,"[""accounts payables"", ""accounts receivables"", ...","[""computer applications specialist certificate...","[""accountant"", ""accounts receivable clerk"", ""m..."
4,0.650000,professional accountant with an outstanding wo...,senior ios engineer bachelor of science (bsc) ...,professional accountant with an outstanding wo...,32,1,5,4.0,NaN,NaN,0.0,senior ios engineer,bachelor of science (bsc) in computer science,ios\r\nios app developer\r\nios application de...,"[""analytical reasoning"", ""compliance testing k...","[""bachelor of business administration""]","[""staff accountant"", ""senior accountant"", ""tax..."


In [12]:
df["combined_text"].iloc[0]

'big data analytics working and database warehouse manager with robust experience in handling all kinds of data. i have also used multiple cloud infrastructure services and am well acquainted with them. currently in search of role that offers more of development. big data hadoop hive python mapreduce spark java machine learning cloud hdfs yarn core java data science c++ data structures dbms rdbms informatica talend amazon redshift microsoft azure b.tech electronics big data analyst technical support troubleshooting collaboration documentation system monitoring software deployment training & mentorship industry trends field visits senior software engineer b.sc in computer science & engineering from a reputed university. at least 1 year'

In [13]:
# everything necessary is in the combined text column
df_relative = df["combined_text"]

In [10]:
# load data
# same preprocessing as linear regression initial test

df = pd.read_csv('../resume_data_cleaned.csv')

# remove bom character and whitespace from column names
df.columns = df.columns.str.strip().str.replace('﻿', '', regex=False)

# drop duplicate column
df = df.drop(columns=['responsibilities.1'])

# strip leading/trailing whitespace from all string values
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# normalize all string values to lowercase
df = df.map(lambda x: x.lower() if isinstance(x, str) else x)

# replace empty strings and n/a with None
df = df.replace({'': None, 'n/a': None})

In [11]:
# select features and target

# columns to use as features
feature_cols = ['skills', 'job_position_name', 'educationaL_requirements', 'responsibilities']

# drop rows where any feature or target is missing
df_model = df[feature_cols + ['matched_score']].dropna()

# combine all text features into a single string per row
df_model['combined'] = df_model[feature_cols].apply(lambda row: ' '.join(row.values), axis=1)

print(f"rows after dropping missing: {df_model.shape[0]}")

rows after dropping missing: 9488


In [41]:
df_model['combined'].iloc[0]

"['big data', 'hadoop', 'hive', 'python', 'mapreduce', 'spark', 'java', 'machine learning', 'cloud', 'hdfs', 'yarn', 'core java', 'data science', 'c++', 'data structures', 'dbms', 'rdbms', 'informatica', 'talend', 'amazon redshift', 'microsoft azure'] senior software engineer b.sc in computer science & engineering from a reputed university. technical support\r\ntroubleshooting\r\ncollaboration\r\ndocumentation\r\nsystem monitoring\r\nsoftware deployment\r\ntraining & mentorship\r\nindustry trends\r\nfield visits"

In [39]:
# code taken from CS4120 Assignment 2
# relativise the word embeddings file to only keep relevant words (words that appear in the training set)
def relativize(file, outfile, word_counter):
    """
    Relativize the word vectors to the given dataset represented by word counts
    :param file: word vectors file
    :param outfile: output file
    :param word_counter: Counter of words occurring in train/dev/test data
    :return:
    """
    f = open(file, encoding="utf-8")
    o = open(outfile, 'w')
    voc = []
    for line in f:
        word = line[:line.find(' ')]
        if word_counter[word] > 0:
            # print("Keeping word vector for " + word)
            voc.append(word)
            o.write(line)
    for word in word_counter:
        if word not in voc:
            count = word_counter[word]
            if count > 1:
                print("Missing " + word + " with count " + repr(count))
    f.close()
    o.close()


def relativize_sentiment_data():
    # Count all words in the train, dev, and *test* sets. Note that this use of looking at the test set is legitimate
    # because we're not looking at the labels, just the words, and it's only used to cache computation that we
    # otherwise would have to do later anyway.
    word_counter = Counter()
    for ex in df_relative:
        for word in re.split("\s|/|\(|-", ex): # split on spaces, /, (, and -
            word = word.strip("()./,-:;•[]!*") # remove characters from start or end of string
            if word == "": # if an empty string is created
                continue # pass on
            word_counter[word] += 1
    # Uncomment these to relativize vectors to the dataset
    relativize("embeddings/dolma_300_2024_1.2M.100_combined.txt", "embeddings/dolma_300_relativised.txt", word_counter)

In [40]:
relativize_sentiment_data()

Missing ieltsinternet with count 341
Missing bachelor’s with count 682
Missing pytesseract with count 56
Missing bnbc with count 342
Missing company's with count 168
Missing pc's with count 28
Missing 10key with count 28
Missing b.ca with count 84
Missing m.ca with count 28
Missing 12+ with count 28
Missing amazon's with count 28
Missing 150+ with count 28
Missing sr.officer with count 341
Missing 2+ with count 28
Missing computersavvy with count 28
Missing b.s.i.t with count 28
Missing shipbd with count 56
Missing tr069 with count 28
Missing 15+ with count 369
Missing python,react with count 340
Missing kpi's with count 28
Missing dvp&r with count 28
Missing master's with count 196
Missing cbmet with count 28
Missing ct's with count 28
Missing amx4+ with count 28
Missing webscrapping with count 28
Missing solutions.â with count 28
Missing 10+ with count 28
Missing i'm with count 56
Missing emerytus with count 28
Missing nanovision with count 28
Missing cmpro with count 28
Missing p.m.

The above print statement shows the strings in the training set that do not appear in the embeddings. The embeddings that I downloaded are from https://nlp.stanford.edu/projects/glove/ and used the newest DOLMA embeddings with 220B tokens and 1.2 million words in the vocab. Once relativised, the file drops from 4.5GB to ~13 MB in size. After splitting the strings and removing excess, I have found a number of words that don't appear in the embeddings. Some of these are useless (see "bacons's") and some would be useful to have ("bachelor's" or "businessintelligence"). This shows that pretrained embeddings probably aren't the best use for resume screening because of the rarety of some words in common language.